In [1]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from scipy.stats import friedmanchisquare
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
import pandas as pd
import numpy as np
import copy
import time

In [2]:
# Reading spambase dataset
dataset = pd.read_csv('/content/spam/spambase.data', header=None)
# spliting data into feature (x) and spam/ham value (y)
xdata = np.array(dataset)[:, :-1]
ydata = np.array(dataset)[:, -1]

In [3]:
#classifier models (Naive Bayes, Decision Tree, k Nearest Neighbour)
NBclf = GaussianNB()
DTclf = DecisionTreeClassifier()
KNNclf = KNeighborsClassifier()

In [4]:
# 10-Fold cross validation classification
splitTen = StratifiedKFold(n_splits=10,shuffle=True)
# define matrix that store results of algorithm and k-fold
accMeas = []
tCompx = []
fMeas = []
for trainIdx, testIdx in splitTen.split(xdata,ydata):
    xtrain, xtest = xdata[trainIdx], xdata[testIdx]
    ytrain, ytest = ydata[trainIdx], ydata[testIdx]
    ACCfold = []
    Tfold = []
    Ffold = []
    # calculate training time of algorithms
    # Naive bayes
    startTime = time.time()
    NBclf.fit(xtrain, ytrain)
    endTime = time.time()
    NBtime = endTime - startTime
    # decision tree
    startTime = time.time()
    DTclf.fit(xtrain, ytrain)
    endTime = time.time()
    DTtime = endTime - startTime
    # Nearest Neighbour
    startTime = time.time()
    KNNclf.fit(xtrain, ytrain)
    endTime = time.time()
    KNNtime = endTime - startTime
    # calculate the accuracy measure 
    NBacc = NBclf.score(xtest, ytest)
    DTacc = DTclf.score(xtest, ytest)
    KNNacc = KNNclf.score(xtest, ytest)
    # calculate f-measure of each model
    NBypred = NBclf.predict(xtest)
    NBf = f1_score(ytest, NBypred)
    DTypred = DTclf.predict(xtest)
    DTf = f1_score(ytest, DTypred)
    KNNypred = KNNclf.predict(xtest)
    KNNf = f1_score(ytest, KNNypred)
    # Save all the comparison results 
    ACCfold.append(NBacc)
    Ffold.append(NBf)
    Tfold.append(NBtime)
    ACCfold.append(DTacc)
    Ffold.append(DTf)
    Tfold.append(DTtime)
    ACCfold.append(KNNacc)
    Ffold.append(KNNf)
    Tfold.append(KNNtime)
    accMeas.append(ACCfold)
    tCompx.append(Tfold)
    fMeas.append(Ffold)
    

In [5]:
# friedman measure table
def friedmanMeasureTable(dta, rowIdx, colName, colSize):
    size_list = list(range(0,colSize))
    data_frame = pd.DataFrame(dta)
    # calculating standard deviation and mean of column
    mean = data_frame.iloc[:,size_list].mean().values.tolist()
    stdev = data_frame.iloc[:,size_list].std().values.tolist()
    data_frame = pd.DataFrame(dta, index=rowIdx)
    data_frame.loc['---------'] = ['|----------|','|----------|','|----------|']
    data_frame.loc['avg=     '] = mean
    data_frame.loc['stdev=   '] = stdev
    data_frame.columns = colName
    return data_frame

In [6]:
def friedmanTable(dta, rowIdx, colName, colSize, istime):
    listOfSize = list(range(0,colSize))
    # generate rank matrix and friedman test table
    RankingData = copy.deepcopy(dta)
    # generating the rank number value
    if istime == 0:
        for r in RankingData:
            sr = sorted(enumerate(r), key=lambda x: x[1])
            idx = [i[0] for i in sr]
            for index, orIdx in enumerate(idx):
                r[orIdx] = len(idx) - index
    else:
        for r in RankingData:
            sr = sorted(enumerate(r), key=lambda x: x[1])
            idx = [i[0] for i in sr]
            for index, orIdx in enumerate(idx):
                r[orIdx] = index + 1

    data_frame = pd.DataFrame(RankingData)
    mean =  data_frame.iloc[:,listOfSize].mean().values.tolist()
    data_frame = pd.DataFrame(RankingData, index=rowIdx)
    data_frame.loc['---------'] = ['|----------|','|----------|','|----------|']
    data_frame.loc['avg Rank '] = mean
    data_frame.columns = colName
    return  data_frame, mean

In [7]:
# calculating friedman p-value
def friedmanPValue(dta, state, perStr):
    dtaNp = np.array(dta)
    q, p = friedmanchisquare(dtaNp[:,0], dtaNp[:,1], dtaNp[:,2])
    if q < state:
        print('Friedman score of Q = ', round(q,3) , ' < ', round(state,3))
        print('No significant distince between',perStr, 'null hypothesis fail to reject thus, algorithms performed equally well.')
        print("----------------------------------------------------------------------------------------")
    else:
        print('Friedman score of Q = ', round(q,3), ' > ', round(state,3))
        print('Significant distince exists between',perStr,'null hypothesis is reject.')
        print("----------------------------------------------------------------------------------------")

In [8]:
# nemenyi test between different algorithm set
def generatingTestNemenyi(mean, meas, colName, k_fold, k):
    alphaQ = 2.343
    print('alpha at significance level 0.05 and k = 3, alphaQ = ', alphaQ)
    criticalDiff = alphaQ * np.sqrt(k * (k + 1.) / (6. *  k_fold))
    print('Calculated critical Different =', round(criticalDiff,4))
    print("----------------------------------------------------------------------------------------")
    set1 = [mean[0], mean[1], 0, 1]
    set2 = [mean[0], mean[2], 0, 2]
    set3 = [mean[1], mean[2], 1, 2]
    for s in [set1, set2, set3]:
        print()
        diff = abs(s[0] - s[1])
        if diff >= criticalDiff:
           print('Rank difference: ',colName[s[2]],'=', round(diff,4), ' >',colName[s[3]],'=',round(criticalDiff,4))
           print( meas+': Significant difference between', colName[s[2]], 'and', colName[s[3]])
           print("----------------------------------------------------------------------------------------")
        else:
            print('Rank difference: ',colName[s[2]],'=', round(diff,4), ' < ',colName[s[3]],'=',round(criticalDiff,4))
            print(meas+': No Significant difference between', colName[s[2]], 'and', colName[s[3]])
            print("----------------------------------------------------------------------------------------")

In [9]:
# running all steps 2,3 and 4
def RunningAllSteps(dta, kfold, colName, colSize, timecompx, meas):
    # Step-2 performance measure friedman table----(Table 12.4)            
    data_frame = friedmanMeasureTable(dta, kfold, colName, colSize)
    print( '\n***********Table gives the possible result of  algorithms for each 10-fold cross-validation on',meas,'----e.g.Table 12.4 ***********')
    print("----------------------------------------------------------------------")
    print(data_frame)
    print('\n')
    # Step-3 performance measure friedman rank table (Table 12.8)
    data_frame, meanRank = friedmanTable(dta, kfold, colName, colSize, timecompx)
    print('***********Table gives the Friedman test rank on', meas, '----e.g.Table 12.8 ***********')
    print("----------------------------------------------------------------------")
    print(data_frame)
    print('\n')
    # Step-4 calculate p-value of friedman test by using average ranking data of each algorithm  
    state = 7.82
    print('***********The Friedman Test Result on',meas,'are given below:*****************\n')
    print("----------------------------------------------------------------------------------------")
    friedmanPValue(accMeas,state,meas)
    print('\n')
    # use nemenyi post-hoc to test if there's signficant difference between algorithm pairs
    generatingTestNemenyi(meanRank, meas, colName, 10,3)
    print('\n')
    return

In [10]:
colName = [' Naive Bayes', ' Decision tree',' Nearest neighbour']
colSize = 3
k_fold = []
for i in range(1,11):
  #f is fold
    idx = 'F' +str(i) 
    k_fold.append(idx)
# perform a comparison between the selected algorithms 
# computational performance in terms of training time
RunningAllSteps(tCompx, k_fold, colName, colSize, 1, 'Time Consumption')


***********Table gives the possible result of  algorithms for each 10-fold cross-validation on Time Consumption ----e.g.Table 12.4 ***********
----------------------------------------------------------------------
            Naive Bayes  Decision tree  Nearest neighbour
F1             0.004256       0.074822           0.000994
F2             0.005998       0.094402            0.00217
F3             0.011175       0.099661           0.000978
F4             0.006398       0.108617           0.001523
F5             0.004879       0.094089           0.000971
F6             0.004721       0.090662           0.001035
F7             0.004773       0.096919           0.000944
F8             0.004874       0.100489           0.000952
F9             0.004798       0.096766           0.000959
F10            0.004946       0.100129           0.001013
---------  |----------|   |----------|       |----------|
avg=           0.005682       0.095656           0.001154
stdev=         0.002033       0

In [11]:
# predictive performance based on accuracy
RunningAllSteps(accMeas, k_fold, colName, colSize, 0, 'Accuracy')


***********Table gives the possible result of  algorithms for each 10-fold cross-validation on Accuracy ----e.g.Table 12.4 ***********
----------------------------------------------------------------------
            Naive Bayes  Decision tree  Nearest neighbour
F1             0.852495       0.911063           0.800434
F2              0.81087       0.891304           0.765217
F3              0.83913       0.921739           0.813043
F4             0.815217       0.926087           0.808696
F5             0.823913       0.934783           0.817391
F6                  0.8        0.93913           0.793478
F7             0.795652       0.919565           0.791304
F8              0.81087       0.917391           0.834783
F9             0.830435       0.904348           0.828261
F10            0.817391       0.915217           0.808696
---------  |----------|   |----------|       |----------|
avg=           0.819597       0.918063            0.80613
stdev=         0.017428       0.014005 

In [12]:
# predictive performance based on F-measure.
RunningAllSteps(fMeas, k_fold, colName, colSize, 0, 'F-measure')


***********Table gives the possible result of  algorithms for each 10-fold cross-validation on F-measure ----e.g.Table 12.4 ***********
----------------------------------------------------------------------
            Naive Bayes  Decision tree  Nearest neighbour
F1             0.839623       0.891247           0.743017
F2             0.799076       0.865591           0.684211
F3             0.824645       0.902703           0.763736
F4             0.805492       0.907104           0.758242
F5             0.808511       0.916667           0.755814
F6             0.789954       0.920904           0.732394
F7             0.785388       0.898072           0.736264
F8             0.798144       0.895604           0.786517
F9             0.817757       0.877778           0.776204
F10            0.803738       0.893733           0.762162
---------  |----------|   |----------|       |----------|
avg=           0.807233        0.89694           0.749856
stdev=         0.016346       0.016652